In [ ]:
# ============================================================
# 0. Imports
# ============================================================
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import (
    Conv2D, MaxPooling2D, GlobalAveragePooling2D,
    Dense, Dropout, BatchNormalization, Input
)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.applications.densenet import preprocess_input as densenet_preprocess
from sklearn.utils import class_weight
import matplotlib.pyplot as plt
import numpy as np
from skimage import exposure
from PIL import Image
import cv2
import os

# AdamW (TF >= 2.11 : tf.keras.optimizers.AdamW ; sinon tensorflow_addons)
try:
    from tensorflow.keras.optimizers import AdamW
except ImportError:
    from tensorflow.keras.optimizers.experimental import AdamW


In [ ]:
# 1. Configuration des chemins
train_dir = os.path.join('train')
test_dir = os.path.join('test')

In [ ]:
# ============================================================
# 1.5 Prétraitement : CROP CENTRÉ + CLAHE
# ----
# Le crop centré (10%) supprime les marqueurs R/L et le texte machine
# qui faisaient dévier l'attention du modèle hors des poumons (effet Clever Hans).
# CLAHE améliore le contraste local des structures pulmonaires.
# Sortie : RGB 3 canaux (grayscale dupliqué) pour compatibilité DenseNet121.
# ============================================================
CROP_RATIO = 0.10   # coupe 10% de chaque côté

def crop_center(img, ratio=CROP_RATIO):
    """Crop centré : supprime `ratio` des 4 bords."""
    h, w = img.shape[:2]
    ch, cw = int(h * ratio), int(w * ratio)
    return img[ch:h-ch, cw:w-cw]

def preprocess_xray(image):
    """
    1) clip + uint8
    2) crop centré (supprime marqueurs R/L)
    3) resize 224x224
    4) CLAHE sur le canal gris
    5) duplication 3 canaux + densenet preprocess_input
    """
    image = np.clip(image, 0, 255).astype(np.uint8)

    # Si l'image est RGB (ImageDataGenerator en color_mode='rgb'),
    # on passe en gris pour appliquer CLAHE puis on redupliquera
    if image.ndim == 3 and image.shape[2] == 3:
        gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    elif image.ndim == 3 and image.shape[2] == 1:
        gray = image[:, :, 0]
    else:
        gray = image

    # Crop centré
    gray = crop_center(gray, CROP_RATIO)

    # Resize à 224x224 après crop
    gray = cv2.resize(gray, (224, 224), interpolation=cv2.INTER_AREA)

    # CLAHE (contraste adaptatif)
    gray_f = exposure.equalize_adapthist(gray, clip_limit=0.02).astype(np.float32)
    gray_u8 = (gray_f * 255.0).astype(np.uint8)

    # Duplication 3 canaux puis preprocess DenseNet (ImageNet mean/std)
    rgb = np.stack([gray_u8, gray_u8, gray_u8], axis=-1).astype(np.float32)
    rgb = densenet_preprocess(rgb)
    return rgb

In [ ]:
def count_classes(directory):
    classes = ['NORMAL', 'PNEUMONIA']
    counts = []
    for cls in classes:
        path = os.path.join(directory, cls)
        counts.append(len(os.listdir(path)))
    return counts

# On recupere les comptes
train_counts = count_classes(train_dir)
test_counts = count_classes(test_dir)

# Visualisation
labels = ['Normal', 'Pneumonie']
fig, ax = plt.subplots(1, 2, figsize=(12, 5))

# Graphique Train
ax[0].bar(labels, train_counts, color=['skyblue', 'salmon'])
ax[0].set_title(f'Entrainement (Total: {sum(train_counts)})')

# Graphique Test
ax[1].bar(labels, test_counts, color=['skyblue', 'salmon'])
ax[1].set_title(f'Test/Val (Total: {sum(test_counts)})')

plt.show()

print(f"Ratio Train: {train_counts[1]/train_counts[0]:.2f}x plus de pneumonies")

In [ ]:
# ============================================================
# 3. Flux de données (SANS GAN — on entraîne sur le train original)
# ----
# - target_size=(224,224) mais le crop 10% est appliqué par preprocess_xray
# - color_mode='rgb' pour nourrir DenseNet121 (3 canaux)
# - augmentation classique (rotation, flip, zoom) pour compenser l'imbalance
# ============================================================
BATCH = 32

train_datagen = ImageDataGenerator(
    rotation_range=15,
    width_shift_range=0.08,
    height_shift_range=0.08,
    shear_range=0.1,
    zoom_range=0.15,
    horizontal_flip=True,
    brightness_range=[0.85, 1.15],   # <-- color jitter (cf. CheXNet enhanced 2025)
    preprocessing_function=preprocess_xray,
    fill_mode='nearest'
)

train_generator = train_datagen.flow_from_directory(
    train_dir,                       # <-- dataset original, pas de GAN
    target_size=(224, 224),
    batch_size=BATCH,
    class_mode='categorical',
    color_mode='rgb'
)

test_datagen = ImageDataGenerator(preprocessing_function=preprocess_xray)
validation_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=(224, 224),
    batch_size=BATCH,
    class_mode='categorical',
    color_mode='rgb',
    shuffle=False
)

print("Classes :", train_generator.class_indices)

In [ ]:
# ============================================================
# 5. Modèle : DenseNet121 pré-entraîné (ImageNet) + tête custom
# ----
# Architecture de référence en imagerie médicale (base de CheXNet, Stanford).
# On part des poids ImageNet puis on entraîne en 2 phases.
# ============================================================
def build_densenet_model(num_classes=2):
    base = DenseNet121(
        weights='imagenet',
        include_top=False,
        input_shape=(224, 224, 3)
    )
    base.trainable = False   # Phase 1 : backbone gelé

    inputs = Input(shape=(224, 224, 3))
    x = base(inputs, training=False)
    x = GlobalAveragePooling2D()(x)
    x = Dropout(0.4)(x)
    x = Dense(128, activation='relu')(x)
    x = Dropout(0.3)(x)
    outputs = Dense(num_classes, activation='softmax')(x)

    model = Model(inputs, outputs, name='DenseNet121_Pneumonie')
    return model, base

model, base_model = build_densenet_model(num_classes=2)
print(f"Paramètres totaux    : {model.count_params():,}")
print(f"Paramètres entraînables (phase 1) : {sum([tf.size(v).numpy() for v in model.trainable_weights]):,}")

In [ ]:
# ============================================================
# 5.5 Focal Loss (catégorielle)
# ----
# Mieux que categorical_crossentropy + class_weight pour un dataset
# déséquilibré (ratio 2.7:1). Donne plus de poids aux exemples difficiles.
# ============================================================
def categorical_focal_loss(gamma=2.0, alpha=0.25):
    def loss(y_true, y_pred):
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)
        ce = -y_true * tf.math.log(y_pred)
        weight = alpha * tf.math.pow(1.0 - y_pred, gamma)
        return tf.reduce_sum(weight * ce, axis=-1)
    return loss

focal = categorical_focal_loss(gamma=2.0, alpha=0.25)